## Configuración y Entorno

In [1]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import StandardScaler

## Ingesta y Limpieza de Datos

In [2]:
# 1. Carga de datos
df =pd.read_csv('customer-segmentation-analysis/data/raw/marketing_campaign.csv', sep='\t')

# 2. Manejo de Valores Nulos
# 'Income' tiene valores faltantes. Usamos la mediana para no afectar la distribución con valores extremos.
df['Income'] = df['Income'].fillna(df['Income'].median())


In [3]:
# 3. Feature Engineering
# Calcular la edad exacta restando el año de nacimiento al año actual (2026)
df['Age'] = 2026 - df['Year_Birth']
# Consolidar todo el gasto en una sola columna 'Total_Spent'
df['Total_Spent'] = df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] + \
                    df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds']

# Consolidar el número de menores en el hogar
df['Total_Children'] = df['Kidhome'] + df['Teenhome']

# Simplificar el estado civil en dos grandes categorías para facilitar el modelado
df['Living_With'] = df['Marital_Status'].replace(
    {'Married': 'Partner', 'Together': 'Partner', 
     'Single': 'Alone', 'Divorced': 'Alone', 'Widow': 'Alone', 
     'Absurd': 'Alone', 'YOLO': 'Alone'}
)


In [4]:
# 4. Limpieza final
# Se eliminan columnas que ya no aportan algun valor tras la transformacion o que son ruido constante
cols_to_drop = ['Year_Birth', 'Marital_Status', 'Kidhome', 'Teenhome', 'Z_CostContact', 'Z_Revenue']
df = df.drop(columns=cols_to_drop)

### Borrado de los valores atipicos
Para esto se realizo una investigacion con graficas de boxplots para detectar que columnas contaban con outliers.
Se determino que en la edad y el ingreso habia valores fuera de lo comun, entonces para edad creamos un limite logico y para los ingresos usamos el rango intercuartilico.

In [5]:
# 1. Limpiar Edad (regla: eliminar mayores de 90 años)
df = df[df['Age'] <= 90]

# 2. Limpiar Ingresos (Método matemático IQR)
Q1 = df['Income'].quantile(0.25)
Q3 = df['Income'].quantile(0.75)
IQR = Q3 - Q1
limite_superior_ingresos = Q3 + 1.5 * IQR

# Filtramos conservando solo los clientes por debajo del límite superior
df = df[df['Income'] <= limite_superior_ingresos]


### Conversion de Variables
Antes de el procesamiento tenemos que traducir los datos para el modelo, de tal manera que la totalidad de columnas tengan datos numericos.

In [6]:
# 1. Codificación de Variables Categóricas (One-Hot Encoding)
# Esto convierte columnas de texto en columnas binarias (0 y 1).
df_encoded = pd.get_dummies(df, drop_first=True)

## Procesamiento / Analisis
Utilizamos la estandarizacion Z-Score para aplastar las variables y que todas tengan el mismo peso de votacion en el modelo.

In [8]:
# Estandarización Matemática
scaler = StandardScaler()

# Ajustamos la escala matemática. El resultado es una matriz pura de NumPy.
matriz_escalada = scaler.fit_transform(df_encoded)

# Lo devolvemos a un formato de tabla Pandas para no perder los nombres de las columnas
df_scaled = pd.DataFrame(matriz_escalada, columns=df_encoded.columns)

### Creacion del nuevo Dataset

In [11]:
# Exportar la tabla original limpia (para entender los resultados después)
df.to_csv('customer-segmentation-analysis\data\processed\df_clean.csv', index=False)

# Exportar la matriz matemática (para alimentar el modelo)
df_scaled.to_csv('customer-segmentation-analysis\data\processed\df_scaled.csv', index=False)


<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
C:\Users\alvar\AppData\Local\Temp\ipykernel_75232\3583845864.py:2: SyntaxWarning: invalid escape sequence '\d'
  df.to_csv('customer-segmentation-analysis\data\processed\df_clean.csv', index=False)
C:\Users\alvar\AppData\Local\Temp\ipykernel_75232\3583845864.py:5: SyntaxWarning: invalid escape sequence '\d'
  df_scaled.to_csv('customer-segmentation-analysis\data\processed\df_scaled.csv', index=False)
